In [1]:
import polars as pl
import json
from typing import List, Dict

In [2]:
interactions_output_parquet_path = '../data/Beauty_new/inter_new.parquet'
df = pl.read_parquet(interactions_output_parquet_path)

In [3]:
def split_session_by_timestamps(
    df: pl.DataFrame,
    time_cutoffs: List[int],
    output_dir: str = None,
) -> List[Dict[int, List[int]]]:
    
    result_dicts = []
    
    def extract_interval(df_source, start, end=None):
        q = df_source.lazy()
        q = q.explode(["item_ids", "timestamps"])
        
        if end is not None:
            q = q.filter(
                (pl.col("timestamps") >= start) & 
                (pl.col("timestamps") < end)
            )
        else:
            q = q.filter(
                pl.col("timestamps") >= start
            )
            
        q = q.group_by("uid").agg([
            pl.col("item_ids").alias("item_ids")
        ]).sort("uid")
        
        return q.collect()
    
    intervals = []
    current_start = 0
    for cutoff in time_cutoffs:
        intervals.append((current_start, cutoff))
        current_start = cutoff
    # от последнего cutoff до бесконечности
    intervals.append((current_start, None))
    
    for start, end in intervals:
        subset = extract_interval(df, start, end)
        
        json_dict = {}
        for user_id, item_ids in subset.iter_rows():
            json_dict[user_id] = item_ids
        
        result_dicts.append(json_dict)
        
        if output_dir:
            if end is not None:
                filename = f"inter_new_[{start}_{end}).json"
            else:
                filename = f"inter_new_[{start}_inf).json"
            
            filepath = f"{output_dir}/{filename}"
            with open(filepath, 'w') as f:
                json.dump(json_dict, f, indent=2)
            
            print(f"Сохранено: {filepath}")
    
    return result_dicts

In [18]:
global_max_time = df.select(
    pl.col("timestamps").explode().max()
).item()

days_val = 14
window_sec = days_val * 24 * 3600 

cutoff_test_start = global_max_time - window_sec      # T - 2w
cutoff_val_start  = global_max_time - 2 * window_sec  # T - 4w
cutoff_gap_start  = global_max_time - 3 * window_sec  # T - 6w

cutoffs = [
    int(cutoff_gap_start),
    int(cutoff_val_start),
    int(cutoff_test_start)
]

print(f"Cutoffs: {cutoffs}")

split_files = split_session_by_timestamps(
    df, 
    cutoffs, 
    output_dir="../sigir/Beauty_new/splits/raw"
)

names = ["Base", "Week -6", "Week -4", "Week -2"]
for i, d in enumerate(split_files):
    print(f"Part {i} [{names[i]}]: {len(d)} users")

Cutoffs: [1402444800, 1403654400, 1404864000]
✓ Сохранено: ../sigir/Beauty_new/splits/raw/inter_new_[0_1402444800).json
✓ Сохранено: ../sigir/Beauty_new/splits/raw/inter_new_[1402444800_1403654400).json
✓ Сохранено: ../sigir/Beauty_new/splits/raw/inter_new_[1403654400_1404864000).json
✓ Сохранено: ../sigir/Beauty_new/splits/raw/inter_new_[1404864000_inf).json
Part 0 [Base]: 22029 users
Part 1 [Week -6]: 1854 users
Part 2 [Week -4]: 1945 users
Part 3 [Week -2]: 1381 users


In [23]:
def merge_and_save(parts_to_merge, dirr, output_name):
    merged = {}
    print(f"Merging {len(parts_to_merge)} files into {output_name}...")
    
    for part in parts_to_merge:
        # with open(fp, 'r') as f:
        #     part = json.load(f)
        for uid, items in part.items():
            if uid not in merged:
                merged[uid] = []
            merged[uid].extend(items)
            
    out_path = f"{dirr}/{output_name}"
    with open(out_path, 'w') as f:
        json.dump(merged, f)
    print(f"Done: {out_path} (Users: {len(merged)})")

p0, p1, p2, p3 = split_files[0], split_files[1], split_files[2], split_files[3]


--- Creating Experiments ---


In [24]:
EXP_DIR = "../sigir/Beauty_new/splits/exp_data"

# Tiger: P0+P1
merge_and_save([p0, p1], EXP_DIR, "exp_4_inter_tiger_train.json")

# 1. Exp 4.1
# Semantics: P0+P1 (Всё кроме валида и теста)
merge_and_save([p0, p1], EXP_DIR, "exp_4.1_inter_semantics_train.json")

# 2. Exp 4.2
# Semantics: P0 (Короче на неделю, без P1)
merge_and_save([p0], EXP_DIR, "exp_4.2_inter_semantics_train_short.json")

# 3. Exp 4.3
# Semantics: P0+P1+P2 (Видит валидацию)
merge_and_save([p0, p1, p2], EXP_DIR, "exp_4.3_inter_semantics_train_leak.json")

# 4. Test Set (тест всех моделей)
merge_and_save([p3], EXP_DIR, "test_set.json")

# 4. Valid Set (пропуск, имитируется разница трейна и теста чтобы потом дообучать)
merge_and_save([p2], EXP_DIR, "valid_skip_set.json")

print("\nAll done!")

Merging 2 files into exp_4_inter_tiger_train.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/exp_4_inter_tiger_train.json (Users: 22129)
Merging 2 files into exp_4.1_inter_semantics_train.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/exp_4.1_inter_semantics_train.json (Users: 22129)
Merging 1 files into exp_4.2_inter_semantics_train_short.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/exp_4.2_inter_semantics_train_short.json (Users: 22029)
Merging 3 files into exp_4.3_inter_semantics_train_leak.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/exp_4.3_inter_semantics_train_leak.json (Users: 22265)
Merging 1 files into test_set.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/test_set.json (Users: 1381)
Merging 1 files into valid_skip_set.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/valid_skip_set.json (Users: 1945)

All done!


In [25]:
merge_and_save([p0, p1, p2, p3], EXP_DIR, "all_set.json")

Merging 4 files into all_set.json...
✓ Done: ../sigir/Beauty_new/splits/exp_data/all_set.json (Users: 22363)


In [41]:
with open("../data/Beauty/inter_new.json", 'r') as f:
    old_inter_new = json.load(f)

with open("../sigir/Beauty_new/splits/exp_data/exp_4.1_inter_semantics_train.json", 'r') as ff:
    first_sem = json.load(ff)
    
with open("../sigir/Beauty_new/splits/exp_data/exp_4.2_inter_semantics_train_short.json", 'r') as ff:
    second_sem = json.load(ff)
    
with open("../sigir/Beauty_new/splits/exp_data/exp_4.3_inter_semantics_train_leak.json", 'r') as ff:
    third_sem = json.load(ff)
    
with open("../sigir/Beauty_new/splits/exp_data/exp_4_inter_tiger_train.json", 'r') as ff:
    tiger_sem = json.load(ff)

with open("../sigir/Beauty_new/splits/exp_data/test_set.json", 'r') as ff:
    test_sem = json.load(ff)

def check_prefix_match(full_data, subset_data, check_suffix=False):
    """
    check_suffix=True включит режим проверки суффиксов (для теста).
    """
    mismatch_count = 0
    full_match_count = 0
    
    for user, sub_items in subset_data.items():
        if user not in full_data:
            print(f"Юзер {user} не найден в исходном файле!")
            mismatch_count += 1
            continue
            
        full_items = full_data[user]
        
        if not check_suffix:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue
                
            if full_items[:len(sub_items)] == sub_items:
                if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1
                
        else:
            if len(sub_items) > len(full_items):
                mismatch_count += 1
                continue
                
            if full_items[-len(sub_items):] == sub_items:
                 if len(full_items) == len(sub_items):
                    full_match_count += 1
            else:
                mismatch_count += 1

    mode = "СУФФИКСЫ" if check_suffix else "ПРЕФИКСЫ"
    
    if mismatch_count == 0:
        print(f"[{mode}] Все {len(subset_data)} массивов ОК. Полных совпадений: {full_match_count}")
    else:
        print(f"[{mode}] Найдено {mismatch_count} ошибок.")

print("Проверка Train сетов (должны быть префиксами):")
check_prefix_match(old_inter_new, first_sem)
check_prefix_match(old_inter_new, second_sem)
check_prefix_match(old_inter_new, third_sem)
check_prefix_match(old_inter_new, tiger_sem)

print("\nПроверка Test сета (должен быть суффиксом):")
check_prefix_match(old_inter_new, test_sem, check_suffix=True)

print("\n(Контроль) Проверка Test сета как префикса (должна упасть):")
check_prefix_match(old_inter_new, test_sem, check_suffix=False)


Проверка Train сетов (должны быть префиксами):
✅ [ПРЕФИКСЫ] Все 22129 массивов ОК. Полных совпадений: 19410
✅ [ПРЕФИКСЫ] Все 22029 массивов ОК. Полных совпадений: 18191
✅ [ПРЕФИКСЫ] Все 22265 массивов ОК. Полных совпадений: 20982
✅ [ПРЕФИКСЫ] Все 22129 массивов ОК. Полных совпадений: 19410

Проверка Test сета (должен быть суффиксом):
✅ [СУФФИКСЫ] Все 1381 массивов ОК. Полных совпадений: 98

(Контроль) Проверка Test сета как префикса (должна упасть):
❌ [ПРЕФИКСЫ] Найдено 1283 ошибок.
